# Data Skipping

In [ ]:
import os
import random
import time
import json
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"

BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppClass04") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

Let's explore the data created

1. Write into the bronze a new table: hotel_booking_bronze_dataskipping
2. DATA SKIPPING: hotel_booking_bronze_dataskipping
3. Below are detailed explanations of the specific TBLPROPERTIES that control data skipping behavior in Delta Lake.
4. Making queries that will benefit from Data Skipping techniques, combined with Z-ordering and Partition.


In [ ]:
def view_deltalog_stats(bucket_name, object_name):
    # Initialize the MinIO client
    client = Minio(
        "minio:9000",  # Replace with your MinIO server endpoint
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=False  # Set to True if using HTTPS
    )
    
    # Retrieve the object from MinIO
    try:
        response = client.get_object(bucket_name, object_name)
        data = response.read().decode('utf-8')
    finally:
        response.close()
        response.release_conn()
    
    # Process the data as before
    lista_jsons = []
    for linha in data.splitlines():
        linha = linha.strip()
        if linha:
            json_obj = json.loads(linha)
            lista_jsons.append(json_obj)
    
    dicionario_unico = {}
    for item in lista_jsons:
        for chave, valor in item.items():
            if chave == "add":
                if chave in dicionario_unico:
                    if not isinstance(dicionario_unico[chave], list):
                        dicionario_unico[chave] = [dicionario_unico[chave]]
                    dicionario_unico[chave].append(valor)
                else:
                    dicionario_unico[chave] = valor
    
    # Display the stats for the first two entries
    for l in dicionario_unico['add'][0:1]:
        json_obj = json.loads(l['stats'])
        print(f"FILE >>> {l['path']}")
        print(f"Total columns collected statistics: {len(json_obj['minValues'])}")
        print(json.dumps(json_obj, indent=2))


### 1 - Write into the bronze a new table: hotel_booking_bronze_dataskipping

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")

In [ ]:
table_bronze = "hotel_booking_bronze_dataskipping"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"

In [ ]:
# Writing in Delta format
df.write.format("delta") \
    .mode("overwrite") \
    .save(location_bronze)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze}
    USING DELTA
    LOCATION '{location_bronze}'
""")

### 2 - DATA SKIPPING: hotel_booking_bronze_dataskipping

In [ ]:
file = "delta/hotel_booking_bronze_dataskipping/_delta_log/00000000000000000000.json"
view_deltalog_stats(bucket_name=BUCKET_BRONZE, object_name=file)

### 3 - Below are detailed explanations of the specific TBLPROPERTIES that control data skipping behavior in Delta Lake.



In the class 02 about Z-odering, when trying to apply using the column: `credit_card` we get the error below:

![alt text](z-ordering-error.png)

We can solve this by increasing the number of columns for collecting stats:

- `delta.dataSkippingNumIndexedCols`: _Adjusts the number of columns (default 32) for which Delta Lake collects statistics used in data skipping._

or **reordering our columns in the delta table** so that the stats of the first columns are collected.

In [ ]:
len(df.columns)

In [ ]:
df.printSchema()

In [ ]:
spark.sql(f"""
    ALTER TABLE {DATABASE}.{table_bronze}
    SET TBLPROPERTIES (
      'delta.dataSkippingNumIndexedCols' = '36'
    );
""")

#### Rewriting data into the table hotel_booking_bronze_dataskipping

In [ ]:
# Writing in Delta format
df.write.format("delta") \
    .mode("overwrite") \
    .save(location_bronze)

In [ ]:
file = "delta/hotel_booking_bronze_dataskipping/_delta_log/00000000000000000002.json"
view_deltalog_stats(bucket_name=BUCKET_BRONZE, object_name=file)

### 4 - Making queries that will benefit from Data Skipping techniques, combined with Z-ordering and Partition.

In [ ]:
spark.sql(f"SELECT hotel, country FROM {DATABASE}.{table_bronze} WHERE is_canceled = 0 LIMIT 2").show(truncate=False)

In [ ]:
spark.sql(f"SELECT MAX(country) FROM {DATABASE}.{table_bronze}").show(truncate=False)

In [ ]:
spark.sql(f"SELECT MIN(country) FROM {DATABASE}.{table_bronze}").show(truncate=False)

In [ ]:
table_bronze_bad = "hotel_booking_partitioned_bronze_bad"
table_bronze_good = "hotel_booking_partitioned_bronze_good"
location_bronze_bad = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_bad}"
location_bronze_good = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_good}"

table_silver = "hotel_booking_silver_zordering"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}"

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_bad}
    USING DELTA
    LOCATION '{location_bronze_bad}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_good}
    USING DELTA
    LOCATION '{location_bronze_good}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_silver}
    USING DELTA
    LOCATION '{location_silver}'
""")

#### Z-ordering

In [ ]:
%%time

# Before applying Z-ordering by country column
spark.sql(f"""
SELECT
    hotel,
    email
FROM {DATABASE}.{table_silver} VERSION AS OF 0
WHERE 
    country = 'ABW'
LIMIT 10
""").show(truncate=False)

In [ ]:
%%time

# After applying Z-ordering by country column
spark.sql(f"""
SELECT
    hotel,
    email
FROM {DATABASE}.{table_silver}
WHERE 
    country = 'ABW'
LIMIT 10
""").show(truncate=False)

#### Partition

In [ ]:
%%time
spark.sql(f"""
SELECT 
    COUNT(*) 
FROM {DATABASE}.{table_bronze_bad} 
WHERE 
    reservation_status_date = '2015-12-15'
""").show()

In [ ]:
%%time
spark.sql(f"""
SELECT COUNT(*)
FROM {DATABASE}.{table_bronze_good} 
WHERE 
    arrival_date_year = '2015'
    AND arrival_date_month = 'December'
""").show(truncate=False)

In [ ]:
# spark.stop()